# LabID: проверка области крепления шипиков

Для нескольких шипиков (нормальных и «на проверку» по контролю качества) показывает два графика рядом:

1. **дендрит** (окрестность шипика, серый), **шипик** (зелёный), **область крепления** (розовая);
2. **запаянный шипик** отдельно, область крепления розовая.

Область крепления — грани «крышки», которыми этап 1 предобработки S запаивает петлю крепления (`attachment_cap_face_indices`): ровно то, что используют обучающие данные. Заплатка лаборатории перед этим снята конвертером (`src/neuron_model/labid_layout.py`).

**Как это работает на mac.** Тяжёлые выходы полного прогона на Windows сюда не нужны. Ноутбук берёт только маленькую таблицу контроля качества с Windows, чтобы выбрать шипики, и для выбранных (не больше 10) локально конвертирует их ветви и повторяет этапы 1–3 и контроль качества из `datasets/labid`. Без таблицы с Windows шипики берутся с первых ветвей, а статусы — из локального прогона.

Таблицы с Windows (скопировать в `WINDOWS_TABLES`, см. ниже): `mesh_quality.parquet` из `...\\labid_raw_z1\\labid\\preprocessed\\` и `labid_conversion.parquet` из `...\\labid_raw_z1\\labid\\`.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Repository root was not found.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from src.neuron_model import labid_visual_check as lvc

In [ ]:
LABID_ROOT = REPO_ROOT / "datasets" / "labid"                       # raw LabID (the whole dataset is local)
WORK_ROOT = REPO_ROOT / "data" / "interim" / "neuron-model" / "labid-visual-check"   # converted branches + local S outputs
CONFIG = REPO_ROOT / "data" / "processed" / "config" / "spine_preprocessing_labid.yaml"
WINDOWS_TABLES = REPO_ROOT / "data" / "interim" / "neuron-model" / "labid-windows-tables"  # mesh_quality.parquet, labid_conversion.parquet

N_VALID, N_REVIEW = 3, 3   # <= 10 spines in total (mac)
SEED = 0                    # another seed = other spines

In [ ]:
quality_path = WINDOWS_TABLES / "mesh_quality.parquet"
windows = None
if quality_path.exists():
    windows = pd.read_parquet(quality_path)
    print("QC of the full Windows run:", windows["status"].value_counts().to_dict())
    print("needs_review reasons:", windows.loc[windows["status"] == "needs_review", "reason"].value_counts().to_dict())
    picked = lvc.pick_spines(windows, n_valid=N_VALID, n_review=N_REVIEW, seed=SEED)
    keys = list(zip(picked["neuron_id"].astype(str), picked["spine_id"].astype(str)))
    display(picked[["neuron_id", "spine_id", "status", "reason"]])
else:
    print(f"no {quality_path}: spines from the first branches, statuses from the local run")
    keys = lvc.local_candidates(LABID_ROOT, max_spines=N_VALID + N_REVIEW + 4)
print(len(keys), "spines")

In [ ]:
res = lvc.prepare(LABID_ROOT, WORK_ROOT, keys, CONFIG, REPO_ROOT / "datasets")
local_qc = res["qc"].assign(neuron_id=lambda d: d["neuron_id"].astype(str), spine_id=lambda d: d["spine_id"].astype(str))
conv = res["conversion"].assign(spine_id=lambda d: d["spine_id"].astype(str))
local_qc[["neuron_id", "spine_id", "status", "reason"]]

In [ ]:
# without the Windows table: keep N_VALID valid + N_REVIEW needs_review of the local run
if windows is None:
    order = pd.concat([local_qc[local_qc["status"] == "valid"].head(N_VALID), local_qc[local_qc["status"] == "needs_review"].head(N_REVIEW)])
    keys = list(zip(order["neuron_id"], order["spine_id"]))

qc_by_key = {(r["neuron_id"], r["spine_id"]): r for r in local_qc.to_dict("records")}
conv_by_key = {(r["neuron_id"], r["spine_id"]): r for r in conv.to_dict("records")}
win_by_key = {} if windows is None else {(str(r["neuron_id"]), str(r["spine_id"])): r for r in windows.to_dict("records")}

for key in keys:
    record = res["records"].get(key)
    if record is None:
        print("not found:", key)
        continue
    try:
        fig = lvc.spine_figure(record, res["cfg"], local_qc=qc_by_key.get(key), windows_row=win_by_key.get(key), conversion_row=conv_by_key.get(key))
    except FileNotFoundError as exc:
        print(key, exc)
        continue
    fig.show()